# 22 — Review and extract YouTube long-workout batch v2

This notebook performs the first full visual label pass for the two new long videos and then extracts continuous pose landmarks. The combined source duration is **61:59**.

Use a **CPU runtime**. MediaPipe runs on the CPU here; a T4 does not provide enough benefit to justify GPU units.

Workflow:

1. download the two temporary source videos and checksum-pinned draft timelines;
2. build start/middle/end contact sheets for all 42 draft intervals;
3. use the review panel to watch every complete interval and correct its boundaries and labels;
4. audit the first-pass timelines;
5. extract full-video 17-joint landmarks at 15 FPS;
6. download `calorie_youtube_long_workouts_v2_review_landmarks.zip`.

The output remains `use_for_training=false`. It contains first-pass labels and continuous landmarks for QA, not approved training clips. A second independent review and adjudication are still required before a training notebook may slice these landmarks into clips. Raw videos remain temporary and are excluded from the ZIP.

In [ ]:
%pip install -q --upgrade yt-dlp mediapipe==0.10.35 opencv-python-headless pandas==2.2.3 ipywidgets tqdm


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import urllib.request

import cv2
import ipywidgets as widgets
import mediapipe as mp
import numpy as np
import pandas as pd
from IPython.display import Image, Video, clear_output, display
from google.colab import files
from tqdm.auto import tqdm

TARGET_FPS = 15.0
PINNED_DATA_COMMIT = 'b4591d836a6e3a26df0e3add7c0afede30c35d59'
RAW_BASE = f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_DATA_COMMIT}'
WORK_ROOT = Path('/content/calorie_youtube_long_v2_work')
VIDEO_ROOT = WORK_ROOT / 'videos'
DRAFT_ROOT = WORK_ROOT / 'draft_labels'
REVIEW_ROOT = WORK_ROOT / 'first_pass_labels'
CONTACT_ROOT = WORK_ROOT / 'contact_sheets'
OUTPUT_ROOT = Path('/content/calorie_youtube_long_workouts_v2_review_landmarks')
LANDMARK_ROOT = OUTPUT_ROOT / 'continuous_landmarks'
for path in (VIDEO_ROOT, DRAFT_ROOT, REVIEW_ROOT, CONTACT_ROOT, OUTPUT_ROOT, LANDMARK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

SOURCES = [
    {'video_id': 'zxqCV_eQHZY', 'creator': 'Legally Swole', 'stem': 'legally_swole__zxqCV_eQHZY', 'duration_s': 1898},
    {'video_id': 'Fk1wnNX4wQ8', 'creator': 'Daniel Nikolic', 'stem': 'daniel_nikolic__Fk1wnNX4wQ8', 'duration_s': 1821},
]
EXPECTED_SHA256 = {
    'candidates.csv': 'fb8af2597dfb93942bb2512e524352036ac6ea5e6ea2df37c1ddd29c0c78da34',
    'daniel_nikolic__Fk1wnNX4wQ8.labels.csv': 'b9fe2646481090c4832df77c58a2beeb0f32faa95b623230448e116e34307009',
    'legally_swole__zxqCV_eQHZY.labels.csv': '49931d7b225f4d369b719e27530b02a42a049d0500b2f5d5d838035854c9916f',
}
print('Runtime: CPU')
print('Pinned data commit:', PINNED_DATA_COMMIT)
print('Final output:', OUTPUT_ROOT.with_suffix('.zip'))


## Download temporary videos and pinned annotation inputs

The videos are downloaded at a pose-friendly resolution and deleted with the Colab runtime. The notebook never packages or redistributes them. The CSV inputs come from an immutable Git commit and are verified before use.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def pinned_download(relative_url, destination, expected_sha):
    destination = Path(destination)
    if not destination.exists() or sha256(destination) != expected_sha:
        temporary = destination.with_suffix(destination.suffix + '.part')
        temporary.unlink(missing_ok=True)
        urllib.request.urlretrieve(f'{RAW_BASE}/{relative_url}', temporary)
        assert sha256(temporary) == expected_sha, f'Checksum mismatch: {destination.name}'
        temporary.replace(destination)
    assert sha256(destination) == expected_sha
    return destination


candidates_path = pinned_download(
    'ml/data/youtube_long_workouts_v2/candidates.csv',
    WORK_ROOT / 'candidates.csv', EXPECTED_SHA256['candidates.csv']
)
for source in SOURCES:
    label_name = f"{source['stem']}.labels.csv"
    draft_path = pinned_download(
        f'ml/data/youtube_long_workouts_v2/annotations/draft/{label_name}',
        DRAFT_ROOT / label_name, EXPECTED_SHA256[label_name]
    )
    review_path = REVIEW_ROOT / label_name
    if not review_path.exists():
        shutil.copy2(draft_path, review_path)

    video_template = VIDEO_ROOT / f"{source['stem']}.%(ext)s"
    matches = list(VIDEO_ROOT.glob(f"{source['stem']}.*"))
    if not matches:
        subprocess.run([
            'yt-dlp', '--no-playlist',
            '-f', '18/best[height<=480][ext=mp4]/best[height<=480]',
            '--merge-output-format', 'mp4', '-o', str(video_template),
            f"https://www.youtube.com/watch?v={source['video_id']}",
        ], check=True)
        matches = list(VIDEO_ROOT.glob(f"{source['stem']}.*"))
    assert len(matches) == 1, (source['stem'], matches)
    source['video_path'] = matches[0]
    cap = cv2.VideoCapture(str(matches[0]))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    decoded_duration = frames / fps if fps else 0
    assert abs(decoded_duration - source['duration_s']) < 8, (source['video_id'], decoded_duration)
    print(source['video_id'], matches[0].name, f'{matches[0].stat().st_size / 1e6:.1f} MB', f'{decoded_duration:.1f}s')


## Build the visual review material

Each contact sheet shows the beginning, middle, and end of one draft interval. Contact sheets help navigation, but they are not sufficient for approval: use **Play full interval** and watch the complete motion before saving each row.

In [ ]:
def read_frame(video_path, time_s):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, time_s) * 1000.0)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Could not decode {video_path} at {time_s:.2f}s')
    return frame


def make_contact_sheet(video_path, row, output_path):
    start, end = float(row.start_s), float(row.end_s)
    margin = min(1.0, max(0.05, (end - start) * 0.05))
    times = [start + margin, (start + end) / 2.0, max(start + margin, end - margin)]
    panels = []
    for time_s in times:
        frame = read_frame(video_path, time_s)
        height, width = frame.shape[:2]
        resized = cv2.resize(frame, (480, max(1, round(height * 480 / width))))
        cv2.putText(resized, f'{time_s:.1f}s', (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        panels.append(resized)
    sheet = np.concatenate(panels, axis=1)
    cv2.imwrite(str(output_path), sheet)


label_tables = {}
contact_index = []
for source in SOURCES:
    label_path = REVIEW_ROOT / f"{source['stem']}.labels.csv"
    table = pd.read_csv(label_path).fillna('')
    label_tables[source['video_id']] = table
    source_contacts = CONTACT_ROOT / source['video_id']
    source_contacts.mkdir(parents=True, exist_ok=True)
    for index, row in tqdm(list(table.iterrows()), desc=f"Contact sheets {source['video_id']}"):
        output_path = source_contacts / f'{index:03d}.jpg'
        if not output_path.exists():
            make_contact_sheet(source['video_path'], row, output_path)
        contact_index.append({'video_id': source['video_id'], 'row_index': index, 'path': str(output_path)})
pd.DataFrame(contact_index).to_csv(WORK_ROOT / 'contact_sheet_index.csv', index=False)
print('Contact sheets:', len(contact_index))


## First-pass annotation panel

Enter a stable reviewer name. For each of the 42 rows, play the full interval, correct its start/end boundaries and motion label, tick the watched box, then save. Unsupported, compound, ambiguous, resting, setup, and transition motion stays `unknown`. Keep `needs_second_review` enabled whenever a boundary or label is uncertain.

In [ ]:
classes_config = json.loads(urllib.request.urlopen(
    f'{RAW_BASE}/ml/config/classes.json'
).read().decode('utf-8'))
CANONICAL = ['unknown'] + classes_config['exercises']
source_by_id = {item['video_id']: item for item in SOURCES}

reviewer = widgets.Text(description='Reviewer:', placeholder='baris_or_reviewer_id')
video_choice = widgets.Dropdown(options=[(f"{s['creator']} — {s['video_id']}", s['video_id']) for s in SOURCES], description='Video:')
row_choice = widgets.IntSlider(description='Row:', min=0, max=len(label_tables[SOURCES[0]['video_id']]) - 1, continuous_update=False)
start_field = widgets.FloatText(description='Start s:')
end_field = widgets.FloatText(description='End s:')
segment_field = widgets.Dropdown(options=['exercise', 'intro', 'outro', 'transition', 'rest', 'cooldown'], description='Segment:')
fine_field = widgets.Text(description='Fine label:', layout=widgets.Layout(width='800px'))
canonical_field = widgets.Dropdown(options=CANONICAL, description='TCN label:')
confidence_field = widgets.Dropdown(options=['high', 'medium', 'low'], description='Confidence:')
precision_field = widgets.FloatText(description='Precision s:')
notes_field = widgets.Textarea(description='Notes:', layout=widgets.Layout(width='900px', height='90px'))
second_field = widgets.Checkbox(value=True, description='Needs second review')
watched_field = widgets.Checkbox(value=False, description='I watched the complete interval')
play_button = widgets.Button(description='Play full interval', button_style='info')
save_button = widgets.Button(description='Save and next', button_style='success')
previous_button = widgets.Button(description='Previous')
display_box = widgets.Output()
message_box = widgets.Output()


def current_table():
    return label_tables[video_choice.value]


def show_row(*_):
    table = current_table()
    row_choice.max = len(table) - 1
    index = min(row_choice.value, row_choice.max)
    row = table.iloc[index]
    start_field.value = float(row.start_s)
    end_field.value = float(row.end_s)
    segment_field.value = str(row.segment_type)
    fine_field.value = str(row.fine_label)
    canonical_field.value = str(row.current_tcn_label)
    confidence_field.value = str(row.annotation_confidence)
    precision_field.value = float(row.boundary_precision_s)
    notes_field.value = str(row.notes)
    second_field.value = str(row.needs_second_review).lower() == 'true'
    watched_field.value = False
    with display_box:
        clear_output(wait=True)
        print(f"Row {index + 1}/{len(table)} | current status={row.review_status} | label={row.current_tcn_label}")
        display(Image(filename=str(CONTACT_ROOT / video_choice.value / f'{index:03d}.jpg')))


def play_interval(_):
    source = source_by_id[video_choice.value]
    start, end = float(start_field.value), float(end_field.value)
    assert 0 <= start < end <= source['duration_s'] + 2
    preview = WORK_ROOT / 'current_review_clip.mp4'
    preview.unlink(missing_ok=True)
    subprocess.run([
        'ffmpeg', '-loglevel', 'error', '-y', '-ss', str(start), '-to', str(end),
        '-i', str(source['video_path']), '-an', '-vf', 'scale=-2:480',
        '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '30', str(preview),
    ], check=True)
    with display_box:
        clear_output(wait=True)
        display(Video(str(preview), embed=True, html_attributes='controls autoplay'))


def save_row(_):
    with message_box:
        clear_output(wait=True)
        if not reviewer.value.strip():
            print('Enter a stable reviewer name before saving.')
            return
        if not watched_field.value:
            print('Watch the complete interval and tick the watched box first.')
            return
        if not (0 <= start_field.value < end_field.value):
            print('Invalid start/end boundary.')
            return
        if segment_field.value != 'exercise' and canonical_field.value != 'unknown':
            print('Non-exercise segments must map to unknown.')
            return
        table = current_table()
        index = row_choice.value
        updates = {
            'start_s': start_field.value, 'end_s': end_field.value,
            'segment_type': segment_field.value, 'fine_label': fine_field.value.strip(),
            'current_tcn_label': canonical_field.value,
            'use_for_training': 'false',
            'annotation_source': 'full_visual_review_via_notebook_22',
            'boundary_precision_s': precision_field.value, 'notes': notes_field.value.strip(),
            'annotation_confidence': confidence_field.value, 'review_status': 'reviewed',
            'reviewer': reviewer.value.strip(),
            'needs_second_review': str(bool(second_field.value)).lower(),
        }
        for column, value in updates.items():
            table.at[index, column] = value
        source = source_by_id[video_choice.value]
        table.to_csv(REVIEW_ROOT / f"{source['stem']}.labels.csv", index=False)
        completed = int((table.review_status == 'reviewed').sum())
        print(f'Saved. Reviewed {completed}/{len(table)} rows in this video.')
        if index < len(table) - 1:
            row_choice.value = index + 1
        else:
            show_row()


def go_previous(_):
    row_choice.value = max(0, row_choice.value - 1)


video_choice.observe(lambda change: (setattr(row_choice, 'value', 0), show_row()), names='value')
row_choice.observe(show_row, names='value')
play_button.on_click(play_interval)
save_button.on_click(save_row)
previous_button.on_click(go_previous)
display(widgets.VBox([
    reviewer, video_choice, row_choice, display_box,
    widgets.HBox([start_field, end_field, precision_field]),
    widgets.HBox([segment_field, canonical_field, confidence_field]),
    fine_field, notes_field, widgets.HBox([second_field, watched_field]),
    widgets.HBox([previous_button, play_button, save_button]), message_box,
]))
show_row()


## Stop here until all 42 intervals have been watched and saved

The next cell intentionally blocks landmark extraction if any draft row remains or if the timeline has gaps/overlaps. This preserves the review-first workflow.

In [ ]:
AUDIT_SCRIPT = WORK_ROOT / 'audit_timeline_annotations.py'
CLASSES_PATH = WORK_ROOT / 'classes.json'
urllib.request.urlretrieve(f'{RAW_BASE}/ml/scripts/audit_timeline_annotations.py', AUDIT_SCRIPT)
urllib.request.urlretrieve(f'{RAW_BASE}/ml/config/classes.json', CLASSES_PATH)

for source in SOURCES:
    path = REVIEW_ROOT / f"{source['stem']}.labels.csv"
    table = pd.read_csv(path, dtype=str).fillna('')
    assert (table.review_status == 'reviewed').all(), f"Unreviewed rows remain in {path.name}"
    assert (~table.reviewer.isin(['', 'metadata_prefill_only'])).all(), f"Human reviewer missing in {path.name}"
    assert (table.use_for_training.str.lower() == 'false').all()

audit_path = OUTPUT_ROOT / 'first_pass_annotation_audit.json'
subprocess.run([
    'python3', str(AUDIT_SCRIPT), str(REVIEW_ROOT),
    '--classes', str(CLASSES_PATH), '--report', str(audit_path),
], check=True)
audit = json.loads(audit_path.read_text())
assert audit['errors'] == 0
print('First-pass annotation audit passed:', audit['rows'], 'rows;', audit['warnings'], 'warnings')


## Extract continuous landmarks at 15 FPS

The arrays cover each complete source video and retain timestamps plus a per-frame detection mask. They are intentionally not split into labelled training clips in this notebook.

In [ ]:
MODEL_PATH = WORK_ROOT / 'pose_landmarker_full.task'
MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
    'pose_landmarker_full/float16/latest/pose_landmarker_full.task'
)
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print('Pose model MB:', round(MODEL_PATH.stat().st_size / 1e6, 2))

KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    scale = max(
        float(np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])),
        float(np.linalg.norm(shoulder_center - hip_center)), 1e-6,
    )
    return np.concatenate([(xyz - hip_center) / scale, frame[:, 3:4]], axis=1)


def fill_missing(sequence):
    shape = sequence.shape
    flat = sequence.reshape(shape[0], -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction='both').fillna(0).to_numpy()
    return filled.reshape(shape).astype(np.float32)


def extract_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open {video_path}')
    source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    next_sample, frame_index = 0.0, 0
    poses, timestamps, detected = [], [], []
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, num_poses=1,
        min_pose_detection_confidence=0.5, min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5, output_segmentation_masks=False,
    )
    progress = tqdm(total=source_frames, desc=Path(video_path).stem, unit='frames')
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            time_s = frame_index / source_fps
            frame_index += 1
            progress.update(1)
            if time_s + 1e-9 < next_sample:
                continue
            next_sample += 1.0 / TARGET_FPS
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(image, int(round(time_s * 1000)))
            if result.pose_landmarks:
                points = result.pose_landmarks[0]
                pose = np.asarray([[points[i].x, points[i].y, points[i].z, float(points[i].visibility or 0.0)] for i in KEEP_17], dtype=np.float32)
                poses.append(normalize_pose(pose))
                detected.append(True)
            else:
                poses.append(np.full((17, 4), np.nan, dtype=np.float32))
                detected.append(False)
            timestamps.append(time_s)
    progress.close()
    cap.release()
    if not poses:
        raise RuntimeError(f'No frames decoded from {video_path}')
    return fill_missing(np.stack(poses)), np.asarray(timestamps, dtype=np.float32), np.asarray(detected, dtype=bool), source_fps


In [ ]:
quality_rows = []
for source in SOURCES:
    output_path = LANDMARK_ROOT / f"{source['stem']}.npz"
    if output_path.exists():
        item = np.load(output_path)
        landmarks, timestamps, detected = item['landmarks'], item['timestamps_s'], item['detected']
        source_fps = float(item['source_fps'])
    else:
        landmarks, timestamps, detected, source_fps = extract_video(source['video_path'])
        np.savez_compressed(
            output_path, landmarks=landmarks.astype(np.float32),
            timestamps_s=timestamps.astype(np.float32), detected=detected,
            source_fps=np.float32(source_fps), target_fps=np.float32(TARGET_FPS),
            video_id=source['video_id'], group_id=f"youtube:{source['video_id']}",
        )
    detection_rate = float(detected.mean())
    quality_rows.append({
        'video_id': source['video_id'], 'creator': source['creator'],
        'group_id': f"youtube:{source['video_id']}",
        'landmark_path': str(output_path.relative_to(OUTPUT_ROOT)),
        'sampled_frames': len(landmarks), 'duration_s': float(timestamps[-1]),
        'target_fps': TARGET_FPS, 'source_fps': source_fps,
        'detection_rate': detection_rate,
        'status': 'ok' if detection_rate >= 0.80 else 'needs_pose_review',
        'label_status': 'first_pass_reviewed_needs_independent_second_review',
        'use_for_training': False,
    })
    print(source['video_id'], landmarks.shape, f'detection={detection_rate:.3f}')
quality = pd.DataFrame(quality_rows)
quality.to_csv(OUTPUT_ROOT / 'continuous_landmark_manifest.csv', index=False)
display(quality)
assert len(quality) == 2
assert quality.sampled_frames.sum() > 50000, 'Unexpectedly few landmark frames'


## Package the review evidence and landmarks

The archive includes first-pass CSVs, the annotation audit, continuous landmark arrays, contact sheets, manifest, and checksums. It excludes raw videos. Return the ZIP to the chat for independent review and training-clip preparation.

In [ ]:
shutil.copytree(REVIEW_ROOT, OUTPUT_ROOT / 'first_pass_labels', dirs_exist_ok=True)
shutil.copytree(CONTACT_ROOT, OUTPUT_ROOT / 'contact_sheets', dirs_exist_ok=True)
shutil.copy2(candidates_path, OUTPUT_ROOT / 'candidates.csv')
run_metadata = {
    'notebook': '22_colab_review_extract_youtube_long_v2.ipynb',
    'runtime': 'CPU', 'target_fps': TARGET_FPS,
    'pinned_data_commit': PINNED_DATA_COMMIT,
    'training_allowed': False,
    'next_step': 'independent second review, adjudication, strict audit, then slice training clips',
}
(OUTPUT_ROOT / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2) + '\n')

checksum_rows = []
for path in sorted(p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.name != 'SHA256SUMS.txt'):
    checksum_rows.append(f"{sha256(path)}  {path.relative_to(OUTPUT_ROOT)}")
(OUTPUT_ROOT / 'SHA256SUMS.txt').write_text('\n'.join(checksum_rows) + '\n')

assert not list(OUTPUT_ROOT.rglob('*.mp4')), 'Raw video must not enter the output archive'
assert len(list(LANDMARK_ROOT.glob('*.npz'))) == 2
zip_path = Path(shutil.make_archive(str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT))
print('ZIP:', zip_path)
print('ZIP size MB:', round(zip_path.stat().st_size / 1e6, 2))
print('SHA-256:', sha256(zip_path))
files.download(str(zip_path))
